In [23]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("../data/raw/Elimited_messy_sales_data.csv")
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,530331,22961,JUMBO BAG RED RETROSPOT,37,8/28/2025 17:08,2.08,17013.0,United Kingdom
1,530359,22457,CREAM CUPID HEARTS COAT HANGER,51,10/13/2025 11:48,2.75,13181.0,United Kingdom
2,530400,21212,PACK OF 72 RETROSPOT CAKE CASES,11,12/18/2025 16:59,0.55,17763.0,United Kingdom
3,530024,22720,POPCORN HOLDER,33,9/17/2025 3:55,0.85,17122.0,United Kingdom
4,C530303,22961,JUMBO BAG RED RETROSPOT,-2,8/29/2025 1:40,2.08,15169.0,United Kingdom


In [24]:
df.shape

(528, 8)

In [25]:
df.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='str')

In [26]:
df.dtypes

InvoiceNo          str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
UnitPrice      float64
CustomerID     float64
Country            str
dtype: object

In [27]:
df.isnull().sum()

InvoiceNo       0
StockCode       0
Description     7
Quantity        0
InvoiceDate     0
UnitPrice       0
CustomerID     10
Country         0
dtype: int64

In [28]:
df.duplicated().sum()

np.int64(8)

In [29]:
df[["Quantity", "UnitPrice"]].describe()

,Quantity,UnitPrice
count,528.000000,528.000000
mean,70.212121,3.515265
std,583.141382,3.938864
min,-51.000000,0.000000
25%,14.000000,1.650000
50%,28.000000,2.080000
75%,42.000000,2.950000
max,9999.000000,18.000000


In [30]:
(df["Quantity"] < 0).sum()

np.int64(13)

In [31]:
(df["Quantity"] == 0).sum()

np.int64(4)

In [32]:
(df["UnitPrice"] == 0).sum()

np.int64(5)

In [33]:
df["InvoiceNo"].astype(str).str.startswith("C").sum()

np.int64(12)

In [34]:
df["Country"].unique()

<ArrowStringArray>
['United Kingdom',           'EIRE',        'GERMANY',       'PORTUGAL',
          'SPAIN',      'Australia',       'Portugal',           'Eire',
        'Belgium',          'Spain',        'BELGIUM',    'Switzerland',
         'France', 'UNITED KINGDOM',    'Netherlands',        'Germany',
         'FRANCE']
Length: 17, dtype: str

## Part 1: Data Understanding

- Dataset has 528 rows and 8 columns.
- Columns: InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country.
- InvoiceDate is stored as text, not a real date, this needs to be converted.
- CustomerID is stored as a decimal number because it has missing values.
- Description is missing in 7 rows.
- CustomerID is missing in 10 rows.
- There are 8 exact duplicate rows.
- 13 rows have negative Quantity, but only 12 InvoiceNo values start with "C" (the usual marker for cancelled orders). This means one negative quantity row does not follow the expected pattern and needs a closer look.
- 4 rows have Quantity equal to 0, which is not a valid sale.
- 5 rows have UnitPrice equal to 0, which is also not valid for a real transaction.
- The highest Quantity value is 9999, which is far above the typical range and looks like an error rather than a real order.
- Country names are inconsistent, some appear in normal case (e.g. "Germany") and others in all uppercase (e.g. "GERMANY"), which would cause the same country to be counted separately during analysis.

## Part 2: Data Cleaning

FixingInvoiceDate (convert to real date)

In [35]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["InvoiceDate"].dtype

dtype('<M8[us]')

Removing duplicates

There are 8 exact duplicate rows. Since these represent the same transaction recorded twice, we will remove the extra copies and keep only one of each.

In [36]:
df = df.drop_duplicates()
df.shape

(520, 8)

Handling missing Description

7 rows are missing a Description. Since we cannot guess the product name, and it is only a small number of rows, we will drop these rows rather than risk incorrect analysis.

In [37]:
df = df.dropna(subset=["Description"])
df.shape

(513, 8)

Handling missing CustomerID

10 rows are missing a CustomerID. These sales are still valid, so instead of removing them, we will label the missing values as "Unknown". This keeps them in overall revenue calculations, while making it clear they cannot be included in customer-specific analysis.

In [38]:
df["CustomerID"] = df["CustomerID"].fillna("Unknown")
df["CustomerID"].isnull().sum()

np.int64(0)

Handling cancelled transactions

13 rows have a negative Quantity, which represents cancelled orders. These are not real sales, so we will remove them from the dataset used for revenue analysis.

In [39]:
df = df[df["Quantity"] > 0]
df.shape

(496, 8)

Handling zero UnitPrice

5 rows have a UnitPrice of 0. Instead of removing these rows, we checked whether the same product appears elsewhere in the data with a valid price. All 4 affected products (22383, 84406B, 22698, 21755) have a consistent price in other rows, so we will use that price to fill in the missing ones instead of losing the data.

In [40]:
zero_price_products = df[df["UnitPrice"] == 0]["StockCode"].unique()
zero_price_products

<ArrowStringArray>
['22383', '84406B', '22698', '21755']
Length: 4, dtype: str

In [41]:
df[df["StockCode"].isin(zero_price_products) & (df["UnitPrice"] > 0)][["StockCode", "Description", "UnitPrice"]]

,StockCode,Description,UnitPrice
6,22383,WHITE HANGING HEART T-LIGHT HOLDER,2.95
13,22698,CREAM CUPID HEARTS COAT HANGER,2.75
16,22698,CREAM CUPID HEARTS COAT HANGER,2.75
19,21755,SET 7 BABUSHKA NESTING BOXES,8.50
20,22698,CREAM CUPID HEARTS COAT HANGER,2.75
...,...,...,...
500,84406B,CREAM CUPID HEARTS COAT HANGER,2.75
502,22383,WHITE HANGING HEART T-LIGHT HOLDER,2.95
507,22383,WHITE HANGING HEART T-LIGHT HOLDER,2.95
519,22383,WHITE HANGING HEART T-LIGHT HOLDER,2.95


In [42]:
price_lookup = df[df["UnitPrice"] > 0].groupby("StockCode")["UnitPrice"].agg(lambda x: x.mode()[0])

zero_price_mask = df["UnitPrice"] == 0
df.loc[zero_price_mask, "UnitPrice"] = df.loc[zero_price_mask, "StockCode"].map(price_lookup)

(df["UnitPrice"] == 0).sum()

np.int64(0)

Investigate the unusual values

In [43]:
df[df["Quantity"] > 500]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
79,530309,22383,WHITE HANGING HEART T-LIGHT HOLDER,5000,2025-09-21 12:52:00,2.95,16448.0,United Kingdom
384,530244,71053,WHITE METAL LANTERN,7500,2025-12-12 02:17:00,3.39,15629.0,United Kingdom
485,530140,20725,LUNCH BAG RED RETROSPOT,9999,2025-11-07 19:50:00,1.65,17066.0,United Kingdom


3 rows have very large Quantity values (5000, 7500, 9999), much higher than every other order in the data. Each of these alone would add a huge amount to total revenue, which does not match the rest of the dataset. These look like data entry mistakes, not real orders, so we remove them.

In [44]:
df = df[df["Quantity"] <= 500]
df.shape

(493, 8)

Fixing inconsistent country names
Some country names appear in different letter cases (e.g. "GERMANY" and "Germany"), which would cause the same country to be treated as two separate groups during analysis. We will standardize all country names to proper case.

In [45]:
df["Country"] = df["Country"].str.title()
df["Country"].unique()

<ArrowStringArray>
['United Kingdom',           'Eire',        'Germany',       'Portugal',
          'Spain',      'Australia',        'Belgium',    'Switzerland',
         'France',    'Netherlands']
Length: 10, dtype: str

Saving the cleaned data

In [47]:
df.to_csv("../data/Processed/cleaned_sales.csv", index=False)

In [48]:
os.path.exists("../data/processed/cleaned_sales.csv")

True